In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [ ]:
!pip install mapply

In [ ]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import torch
import mapply
mapply.init(n_workers=-1,progressbar=True)
tqdm.pandas()

class Solver:
    
    def __init__(self,model="facebook/bart-large-mnli"):
        self.model = model
        
    def solve(self, text, labels):
        classifier = pipeline(
            "zero-shot-classification",
            model = self.model
        )
        result = classifier(text, labels)
        return result

In [ ]:
device_id = 0 if torch.cuda.is_available() else -1

In [ ]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [ ]:
def get_top3(prompt, options, solver):

    result = solver.solve(prompt, options)
    winning_option_texts = result['labels']
    keys = ['A', 'B', 'C', 'D', 'E']
    top3_keys = []
    for text in winning_option_texts[:3]:
        original_index = options.index(text)
        top3_keys.append(keys[original_index])
        
    return " ".join(top3_keys)


solver = Solver()
test_df['Prediction'] = test_df.progress_apply(lambda x: get_top3(x['prompt'],[x['A'],x['B'],x['C'],x['D'],x['E']],solver), axis = 1)

In [ ]:
sub_df = test_df[['id','Prediction']]
sub_df.columns = ['ID','Prediction']
sub_df.to_csv('submission.csv',index=False)

In [ ]:
sub_df

In [ ]:
sub_df.Prediction.unique()